In [1]:
from datetime import datetime, timedelta
from utils.spark_session import get_sparkSession 
from utils.utilities import generate_date
from utils.s3_aws import archive_landing

In [2]:
spark = get_sparkSession("TEST AWS")
spark

In [10]:
spark.sql("show databases").show()

+---------+
|namespace|
+---------+
|      bad|
|   bronze|
|  default|
|     gold|
|   silver|
+---------+



In [3]:
spark.sql("create database if not exists test")
spark.sql("drop table if exists test.test_table")
spark.sql("""
create table test.test_table(id STRING,
name STRING
) USING delta
""")
print("Table created")

Table created


In [4]:

_filename = "test.csv"
_file_path = f"s3a://ecommerce-pooja/pipeline/landing/fact_payments/{_filename}"


In [5]:
#Reading csv file & creating dataframe

df = spark.read.format("csv") \
          .option("header", True) \
          .load(_file_path)

df.count()
df.show()

+----+-----+
|  id| name|
+----+-----+
|A100|Test1|
|A200|Test2|
+----+-----+



In [6]:
df.write.format("delta").mode("overwrite").saveAsTable("test.test_table")

In [7]:
spark.read.table("test.test_table").show()

+----+-----+
|  id| name|
+----+-----+
|A100|Test1|
|A200|Test2|
+----+-----+



In [8]:
if archive_landing(_filename, "fact_payments"):
    print(f"SPARK-APP: File {_filename} archived")
else:
    print(f"SPARK-APP: Error while archiving {_filename} from landing")

SPARK-APP: File test.csv archived


In [4]:
spark.stop()